In [2]:
!pip install airportsdata
!pip install torch

   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
    --------------------------------------- 2.6/113.8 MB 12.5 MB/s eta 0:00:09
   -- ------------------------------------- 6.3/113.8 MB 14.8 MB/s eta 0:00:08
   --- ------------------------------------ 9.7/113.8 MB 15.9 MB/s eta 0:00:07
   ---- ----------------------------------- 13.6/113.8 MB 16.4 MB/s eta 0:00:07
   ----- ---------------------------------- 16.3/113.8 MB 15.7 MB/s eta 0:00:07
   ------- -------------------------------- 20.2/113.8 MB 16.3 MB/s eta 0:00:06
   -------- ------------------------------- 24.4/113.8 MB 16.4 MB/s eta 0:00:06
   ---------- ----------------------------- 29.6/113.8 MB 17.6 MB/s eta 0:00:05
   ------------ --------------------------- 34.3/113.8 MB 18.0 MB/s eta 0:00:05
   ------------- -------------------------- 39.6/113.8 MB 18.9 MB/s eta 0:00:04
   --------------- ------------------------ 45.1/113.8 MB 19.5 MB/s eta 0:00:04
   ----------------- ---------------------- 50.1/113

In [1]:
import numpy as np
import pandas as pd
import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler, TargetEncoder
import airportsdata

In [5]:
import os

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler, TargetEncoder
import airportsdata

: 

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [3]:
import time
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, roc_curve, f1_score, precision_recall_curve

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
#We use a custom dataset due to the unique nature of our data. We train on sequences and a single flight, the next in the sequence, whose delay we would like to guess.
#Sequences_cont is all the fields we want to use for the sequence we are training on.
#Target_cont is all the fields we want to use for the single next slight in the sequence.
#The difference is that sequences_cont we use DepDelay and ArrDelay as features but Target_cont we don't.
#This is not cheating, since we are never using the future to predict the past.
class FlightDataset(Dataset):
    def __init__(self, sequences_cont, target_cont, labels):
        self.sequences_cont = torch.tensor(sequences_cont, dtype=torch.float32)
        self.target_cont = torch.tensor(target_cont, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.sequences_cont[idx],
            self.target_cont[idx],
            self.labels[idx]
        )

In [6]:
#Can play around with hidden_size (the number of nodes to train on in the LSTM), the dropout ratio, the structure of nn.classifier
class FlightDelayRNN(nn.Module):
    def __init__(
        self,
        n_cont_features,
        n_target_cont_features,
        dropout = 0.25,
        hidden_size = 32
    ):
        super(FlightDelayRNN, self).__init__()

        lstm_input_size = n_cont_features

        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=hidden_size,
            batch_first=True,
            num_layers = 2
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + n_target_cont_features, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(16, 1)
        )

    def forward(self, x_cont, target_cont):
        output, (h_n, c_n) = self.lstm(x_cont)
        final_hidden = h_n[-1]
        observation = torch.cat([final_hidden, target_cont], dim=-1)
        return self.classifier(observation).squeeze(1)

In [7]:
def train_model(model, train_loader, val_loader, pos_weight):
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=2, factor=0.5
    )

    best_val_loss = float('inf')

    for epoch in range(10):
        print(f'Beginning Epoch {epoch+1}')
        model.train()
        train_losses = []

        #training
        for  x_cont, x_target_cont, labels in train_loader:
            x_cont = x_cont.to(device)
            x_target_cont = x_target_cont.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            predictions = model( x_cont,  x_target_cont)
            loss = criterion(predictions, labels)
            loss.backward()

            # Gradient clipping — important for RNNs
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_losses.append(loss.item())


        model.eval()
        val_losses = []

        #validation
        with torch.no_grad():
            for x_cont, x_target_cont, labels in val_loader:
                x_cont = x_cont.to(device)
                x_target_cont = x_target_cont.to(device)
                labels = labels.to(device)

                predictions = model( x_cont, x_target_cont)
                loss = criterion(predictions, labels)
                val_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)

        scheduler.step(avg_val_loss)

        # save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')

        print(f'Epoch {epoch+1:3d} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {avg_val_loss:.4f}')

    print('Training complete. Best model saved.')

In [ ]:
#for the first ten months of 2023. you can change this around as well.
file_paths = glob.glob('/content/drive/MyDrive/Erdos Project Data/combined-data-fix/*.csv')
df = pd.concat([pd.read_csv(f) for f in file_paths[25:28]], ignore_index=True)

In [27]:
##Loads local data, for the purpose of code tesing
path = os.path.join(r"C:\Users\David\Documents\GitHub\spring-2026-airline-reliability\combined-data", "2023_*.csv")

all_files = glob.glob(path)
df=[]
df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)



In [25]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Year', 'Month', 'DayofMonth',
       'DayOfWeek', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes',
       'DepDel15', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes',
       'ArrDel15', 'Cancelled', 'AirTime', 'WeatherDelay', 'NotWeatherDelay',
       'Dest', 'Origin', 'Reporting_Airline', 'Tail_Number', 'TaxiIn',
       'TaxiOut', 'CancelOrDelayed', 'Holiday', 'dep_longitude',
       'dep_latitude', 'arr_longitude', 'arr_latitude', 'UTC_CRSDepTime',
       'UTC_CRSDepBaseline', 'UTC_CRSDepStep', 'UTC_CRSArrTime',
       'UTC_CRSArrBaseline', 'UTC_CRSArrStep', 'UTC_CRSDepTime_2',
       'UTC_CRSDepBaseline_2', 'UTC_CRSDepStep_2', 'UTC_CRSArrTime_2',
       'UTC_CRSArrBaseline_2', 'UTC_CRSArrStep_2', 'dep_tp', 'dep_mxtpr',
       'dep_fg10', 'dep_mx2t', 'dep_mn2t', 'dep_u10', 'dep_v10', 'arr_tp',
       'arr_mxtpr', 'arr_fg10', 'arr_mx2t', 'arr_mn2t', 'arr_u10', 'arr_v10',
       'dep_tp_2', 'dep_mxtpr_2', 'dep_fg10_2', 'dep_mx2t_2', '

In [31]:
#cont_features - the features we use in our sequence
#target_cont_features - the features we use for the flight we want to predict the delay of
cont_features = ['delta_t', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10',
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10', 'DepDelay', 'ArrDelay','Holiday']
target_cont_features = ['Origin', 'Dest', 'Reporting_Airline', 'Hour_sin','Hour_cos', 'Month_sin','Month_cos', 'DayofMonth', 'delta_t', 'Year', 'CRSDepTime', 'CRSArrTime', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10',
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10', 'avg_delay_prev_hour','Holiday']

In [10]:
from datetime import datetime, timezone, timedelta
import zoneinfo

In [11]:
iata_airports = airportsdata.load('IATA')

In [12]:
# takes in a date and military time at a certain airport and returns the UTC time
# this function is no longer being used
def convert_date(year, month, day, hhmm, airport):
    dt = datetime.strptime(str(hhmm).zfill(4), '%H%M')
    dt = dt.replace(year = year, month=month, day = day, tzinfo = zoneinfo.ZoneInfo(iata_airports[airport]['tz']))
    return dt.astimezone(timezone.utc)

In [13]:
#the next two lines I've commented out should no longer be necessary since Joao added a more correct UTC_CRSDepTime column in the data
#df['UTC_CRSDepTime'] = df.apply(lambda row: convert_date(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['Origin']), axis=1)
#df = df.sort_values(['Tail_Number', 'UTC_CRSDepTime'])
df['UTC_CRSDepTime'] = pd.to_datetime(df['UTC_CRSDepTime'])
df = df.sort_values(by=['Tail_Number', 'UTC_CRSDepTime']).reset_index(drop=True)
df['delta_t'] = (df.groupby('Tail_Number')['UTC_CRSDepTime'].diff().dt.total_seconds() / 3600).fillna(0)

In [14]:
#removing cancelled flights and any rows that have NaN for some reason
df = df[df['Cancelled'] ==0]
df = df.dropna()

In [28]:
df.describe()

,Unnamed: 0.1,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,...,dep_mn2t_2,dep_u10_2,dep_v10_2,arr_tp_2,arr_mxtpr_2,arr_fg10_2,arr_mx2t_2,arr_mn2t_2,arr_u10_2,arr_v10_2
count,8.953613e+06,8.953613e+06,8953613.0,8.953613e+06,8.953613e+06,8.953613e+06,8.953613e+06,8.813613e+06,8.813553e+06,8.813553e+06,...,4.179576e+06,4.179576e+06,4.179576e+06,4.179576e+06,4.179576e+06,4.179576e+06,4.179576e+06,4.179576e+06,4.179576e+06,4.179576e+06
mean,2.729203e+05,2.869984e+05,2023.0,4.786398e+00,1.577040e+01,3.967182e+00,1.335302e+03,1.337699e+03,1.442902e+01,1.737113e+01,...,2.897215e+02,4.585303e-01,3.573116e-01,1.138202e-04,6.061536e-05,7.167721e+00,2.908678e+02,2.901369e+02,4.589828e-01,3.791087e-01
std,1.595869e+05,1.608920e+05,0.0,2.402248e+00,8.778476e+00,2.000132e+00,4.999370e+02,5.177414e+02,5.908814e+01,5.810552e+01,...,1.038684e+01,2.374154e+00,2.506662e+00,5.027584e-04,3.009894e-04,3.218053e+00,1.040805e+01,1.037233e+01,2.392490e+00,2.515866e+00
min,0.000000e+00,0.000000e+00,2023.0,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,-6.800000e+01,0.000000e+00,...,2.380500e+02,-1.384460e+01,-1.454576e+01,0.000000e+00,0.000000e+00,5.060165e-01,2.401584e+02,2.395765e+02,-1.417590e+01,-1.453101e+01
25%,1.358430e+05,1.491320e+05,2023.0,3.000000e+00,8.000000e+00,2.000000e+00,9.070000e+02,9.080000e+02,-5.000000e+00,0.000000e+00,...,2.823709e+02,-1.157089e+00,-1.288136e+00,0.000000e+00,0.000000e+00,4.783085e+00,2.835056e+02,2.827856e+02,-1.180069e+00,-1.281784e+00
50%,2.717930e+05,2.854390e+05,2023.0,5.000000e+00,1.600000e+01,4.000000e+00,1.325000e+03,1.327000e+03,-1.000000e+00,0.000000e+00,...,2.908943e+02,4.540100e-01,3.745728e-01,0.000000e+00,0.000000e+00,6.722707e+00,2.921950e+02,2.914277e+02,4.564972e-01,4.033814e-01
75%,4.080360e+05,4.216850e+05,2023.0,7.000000e+00,2.300000e+01,6.000000e+00,1.745000e+03,1.755000e+03,1.200000e+01,1.200000e+01,...,2.975486e+02,1.961395e+00,1.967483e+00,8.583069e-06,2.145767e-06,9.001969e+00,2.987007e+02,2.979355e+02,1.977478e+00,2.005356e+00
max,5.832370e+05,6.029860e+05,2023.0,9.000000e+00,3.100000e+01,7.000000e+00,2.359000e+03,2.400000e+03,4.413000e+03,4.413000e+03,...,3.205974e+02,1.755881e+01,1.418892e+01,2.333784e-02,8.838892e-03,3.150313e+01,3.208093e+02,3.205974e+02,1.623344e+01,1.418892e+01


In [29]:
#Add holiday column
holidays_2020 = [(1,1),(1,20),(2,17),(5,25),(7,3),(9,7),(10,12),(11,11),(11,26),(12,25)]
holidays_2021 = [(1,1),(1,18),(2,15),(5,31),(6,18),(7,5),(9,6),(10,11),(11,11),(11,25),(12,24)]
holidays_2022 = [(1,1),(1,17),(2,21),(5,30),(6,20),(7,4),(9,5),(10,10),(11,11),(11,24),(12,26)]
holidays_2023 = [(1,2),(1,16),(2,20),(5,29),(6,19),(7,4),(9,4),(10,9),(11,10),(11,23),(12,25)]
holidays_2024 = [(1,1),(1,15),(2,19),(5,27),(6,19),(7,5),(9,2),(10,14),(11,11),(11,28),(12,25)]
holidays_2025 = [(1,1),(1,20),(2,17),(5,26),(6,19),(7,4),(9,1),(10,13),(11,11),(11,27),(12,25)]


holiday_map = {
    2020: holidays_2020,
    2021: holidays_2021,
    2022: holidays_2022,
    2023: holidays_2023,
    2024: holidays_2024,
    2025: holidays_2025
}

df['Year'] = df['Year'].astype(int)
df['Month'] = df['Month'].astype(int)
df['DayofMonth'] = df['DayofMonth'].astype(int)

holiday_tuples = list(zip(df['Month'], df['DayofMonth']))
df['Holiday'] = [
    2 if tup in holiday_map.get(yr, []) else 0 
    for tup, yr in zip(holiday_tuples, df['Year'])
]

base = df['Holiday']
prev1 = base.shift(-1, fill_value=0) # 1 day before
prev2 = base.shift(-2, fill_value=0) # 2 day before
next1 = base.shift(1, fill_value=0) # 1 day after
next2 = base.shift(2, fill_value=0) # 2 day after
     
intersection = (prev1 == 2) | (prev2 == 2) | (next1 == 2) | (next2 == 2)

df['Holiday'] = np.where(base == 2, 2, np.where(intersection, 1, 0))



In [30]:
df['Holiday'].describe()

count    8.953613e+06
mean     9.220155e-02
std      3.658173e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.000000e+00
Name: Holiday, dtype: float64

In [15]:
#no longer being used because it's too slow
def convert_to_local_hour(utc_time, timezone):
    return utc_time.tz_localize('UTC').tz_convert(timezone).hour

In [16]:
#made with LLM, extracts the hour from CRSDepHour
def extract_military_hour(time_val):
    """Extract hour from military time (e.g., 18 -> 0, 1155 -> 11, 2319 -> 23)"""
    # Convert to string and strip whitespace
    time_str = str(time_val).strip()

    if len(time_str) <= 2:
        # Values like '18' or '5' are minutes-only format → hour = 0
        return 0
    elif len(time_str) == 3:
        # Values like '930' → hour = 9
        return int(time_str[0])
    elif len(time_str) == 4:
        # Values like '1155', '2319' → first two digits are the hour
        return int(time_str[:2])

In [17]:
#creates the hour column
df['Hour'] = df['CRSDepTime'].apply(extract_military_hour) #df.apply(
    #lambda row: convert_to_local_hour(row['UTC_CRSDepTime'], zoneinfo.ZoneInfo(iata_airports[row['Origin']]['tz'])), axis=1
#)

In [18]:
#cyclic hour and month categories
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

In [19]:
def utc_exact(year, month, day, hhmm, airport):
    if hhmm == 2400:
        dt = datetime.strptime('0000', '%H%M')
        dt = dt.replace(year = year, month = month, day = day, tzinfo = zoneinfo.ZoneInfo(iata_airports[airport]['tz']))
        dt = (dt + timedelta(days = 1))
    else:
        dt = datetime.strptime(str(hhmm).zfill(4), '%H%M')
        dt = dt.replace(year = year, month = month, day = day, tzinfo = zoneinfo.ZoneInfo(iata_airports[airport]['tz']))
    dt = dt.astimezone(timezone.utc)
    dt = dt.replace(tzinfo = None)

    return dt

In [20]:
#TargetEncoder() was really slow so this (LLM-generated) manually target encodes
def target_encode(train_df, val_df, cat_cols, target_col):
    """
    Manually compute target encoding using pandas groupby mean.
    Much faster than sklearn TargetEncoder for large dataframes.
    """
    encoding_maps = {}

    for col in cat_cols:
        # Compute mean target per category on train only
        means = train_df.groupby(col)[target_col].mean()
        global_mean = train_df[target_col].mean()  # fallback for unseen categories

        encoding_maps[col] = means

        # Apply to train and val
        train_df[col] = train_df[col].map(means).fillna(global_mean)
        val_df[col]   = val_df[col].map(means).fillna(global_mean)

    return train_df, val_df, encoding_maps


In [23]:
#splits up to make a training and validation set. should be adjusted based on whatever time period you're looking at.
train_df = df[(df['Month'] < 2)].copy()
val_df = df[(df['Month'] > 1)].copy()

#fields that you want to TargetEncode
cat_cols_to_encode = ['Origin', 'Dest', 'Reporting_Airline']

train_df['UTC_CRSDepExact'] = train_df.apply(lambda row: utc_exact(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['Origin']),
                                          axis = 1)
val_df['UTC_CRSDepExact'] = val_df.apply(lambda row: utc_exact(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['Origin']),
                                          axis = 1)

train_df = train_df.sort_values(
    ['UTC_CRSDepExact', 'Origin'],
    ascending=True
).reset_index(drop=True)

train_df = train_df.set_index('UTC_CRSDepExact')

train_df['avg_delay_prev_hour'] = (
    train_df.groupby('Origin')['DepDel15']
    .transform(lambda row: row.rolling('1h', closed='left', min_periods = 0).mean().fillna(0))
)

val_df = val_df.sort_values(
    ['UTC_CRSDepExact', 'Origin'],
    ascending=True
).reset_index(drop=True)

val_df = val_df.set_index('UTC_CRSDepExact')

val_df['avg_delay_prev_hour'] = (
    val_df.groupby('Origin')['DepDel15']
    .transform(lambda row: row.rolling('1h', closed='left', min_periods = 0).mean().fillna(0))
)

train_df, val_df, encoding_maps = target_encode(train_df, val_df, cat_cols_to_encode, 'DepDel15')

scaler_cont = StandardScaler()
scaler_target = StandardScaler()

train_df[cont_features] = scaler_cont.fit_transform(train_df[cont_features])
train_df[target_cont_features] = scaler_target.fit_transform(train_df[target_cont_features])

val_df[cont_features] = scaler_cont.transform(val_df[cont_features])
val_df[target_cont_features] = scaler_target.transform(val_df[target_cont_features])

In [ ]:
#frees memory ideally
del df

In [24]:
#returns sequences of a certain length, and the next flight in the sequence, and that next flight's label which here is DepDelayMinutes
def build_sequences(data, sequence_length):
    X_cont_list = []
    X_target_cont_list = []
    y_list = []

    for tail, group in data.groupby('Tail_Number'):
        group = group.sort_values('UTC_CRSDepTime').reset_index(drop=True)

        cont_values = group[cont_features].to_numpy(dtype=float)
        target_cont_values = group[target_cont_features].to_numpy(dtype=float)
        labels = group['DepDel15'].to_numpy()

        for i in range(len(group) - sequence_length):
            X_cont_list.append(cont_values[i : i + sequence_length])
            X_target_cont_list.append(target_cont_values[i+sequence_length])

            y_list.append(labels[i + sequence_length])

    X_cont = np.array(X_cont_list)
    X_target_cont = np.array(X_target_cont_list)
    y = np.array(y_list)

    return X_cont, X_target_cont, y


In [25]:
#runs the model, you can modify sequence_length here.
X_cont_train, X_target_cont_train, y_train = build_sequences(train_df, sequence_length=5)
X_cont_val, X_target_cont_val, y_val = build_sequences(val_df, sequence_length=5)

pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)

train_dataset = FlightDataset(X_cont_train, X_target_cont_train, y_train)
val_dataset = FlightDataset(X_cont_val,X_target_cont_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=256, shuffle=False)

model = FlightDelayRNN(
    n_cont_features = len(cont_features),
    n_target_cont_features = len(target_cont_features)
).to(device)

train_model(model, train_loader, val_loader, pos_weight)

Beginning Epoch 1
Epoch   1 | Train Loss: 0.8302 | Val Loss: 0.8087
Beginning Epoch 2
Epoch   2 | Train Loss: 0.7947 | Val Loss: 0.7971
Beginning Epoch 3
Epoch   3 | Train Loss: 0.7892 | Val Loss: 0.7938
Beginning Epoch 4
Epoch   4 | Train Loss: 0.7845 | Val Loss: 0.7945
Beginning Epoch 5
Epoch   5 | Train Loss: 0.7821 | Val Loss: 0.7892
Beginning Epoch 6
Epoch   6 | Train Loss: 0.7794 | Val Loss: 0.7876
Beginning Epoch 7
Epoch   7 | Train Loss: 0.7774 | Val Loss: 0.7926
Beginning Epoch 8
Epoch   8 | Train Loss: 0.7760 | Val Loss: 0.7992
Beginning Epoch 9
Epoch   9 | Train Loss: 0.7748 | Val Loss: 0.7990
Beginning Epoch 10
Epoch  10 | Train Loss: 0.7701 | Val Loss: 0.7943
Training complete. Best model saved.


In [26]:
def evaluate_model(model, test_loader):
    model.load_state_dict(torch.load('best_model.pt'))
    model.eval()
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for x_cont, x_target_cont, labels in test_loader:
            x_cont = x_cont.to(device)
            x_target_cont = x_target_cont.to(device)
            logits  = model(x_cont, x_target_cont)
            probs   = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_preds  = (all_probs > 0.59).astype(int)

    print(classification_report(all_labels, all_preds,
          target_names=['Delay < 15', 'Delay >= 15']))
    print(f'ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}')
    print(f'accuracy: {accuracy_score(all_labels, all_preds):.4f}')

    return all_probs, all_preds, all_labels

In [27]:
all_probs, all_preds, all_labels = evaluate_model(model, val_loader)

              precision    recall  f1-score   support

  Delay < 15       0.89      0.92      0.90    758865
 Delay >= 15       0.65      0.57      0.61    198659

    accuracy                           0.85    957524
   macro avg       0.77      0.74      0.76    957524
weighted avg       0.84      0.85      0.84    957524

ROC-AUC: 0.8280
accuracy: 0.8468


In [28]:
precision, recall, thresholds = precision_recall_curve(all_labels, all_probs)

# Compute F1 for all thresholds at once using numpy (no loop)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Best threshold (F1): {best_threshold:.4f}")
print(f"Best F1: {f1_scores[best_idx]:.4f}")

all_preds = (all_probs >= best_threshold).astype(int)

Best threshold (F1): 0.5889
Best F1: 0.6068


Below is code for logistic regression as a baseline.

In [29]:
ct = ColumnTransformer(
    [
    ('onehot', OneHotEncoder(handle_unknown='ignore'), ['Tail_Number'])
    ], remainder = 'passthrough'
)

log_reg = Pipeline([
    ('columntransform', ct),
    ('model', LogisticRegression())
])

In [30]:
target = ['DepDel15']
features = ['Origin', 'Dest', 'Reporting_Airline', 'Tail_Number', 'Hour_sin','Hour_cos', 'Month_sin','Month_cos', 'DayofMonth', 'delta_t', 'Year', 'CRSDepTime', 'CRSArrTime', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10',
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10', 'avg_delay_prev_hour']

In [31]:
start_fit_log = time.perf_counter()
log_reg.fit(train_df[features], train_df[target])
end_fit_log = time.perf_counter()
time_fit_log = end_fit_log - start_fit_log
print(f'fit took {time_fit_log}')

start_pred_log = time.perf_counter()
log_pred = log_reg.predict(val_df[features])
end_pred_log = time.perf_counter()
time_pred_log = end_pred_log - start_pred_log
print(f'pred took {time_pred_log}')

print(classification_report(val_df[target], log_pred,
          target_names=['Delay < 15', 'Delay >= 15']))
print(f'ROC-AUC: {roc_auc_score(val_df[target], log_reg.predict_proba(val_df[features])[:, 1]):.4f}')
print(f'accuracy: {accuracy_score(val_df[target], log_pred):.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


fit took 7.812027597999986
pred took 4.161617525999645
              precision    recall  f1-score   support

  Delay < 15       0.81      0.97      0.88    779753
 Delay >= 15       0.52      0.12      0.20    204004

    accuracy                           0.79    983757
   macro avg       0.66      0.55      0.54    983757
weighted avg       0.75      0.79      0.74    983757

ROC-AUC: 0.6814
accuracy: 0.7945


Below is code of the dummy baseline.

In [32]:
from sklearn.dummy import DummyClassifier

In [33]:
ct_dumb = ColumnTransformer(
    [
    ('onehot', OneHotEncoder(handle_unknown='ignore'), ['Tail_Number'])
    ], remainder = 'passthrough'
)

dummy = Pipeline([
    ('columntransform', ct_dumb),
    ('model', DummyClassifier(strategy='most_frequent'))
])

In [34]:
start_fit_dumb = time.perf_counter()
dummy.fit(train_df[features], train_df[target])
end_fit_dumb = time.perf_counter()
time_fit_dumb = end_fit_dumb - start_fit_dumb
print(f'fit took {time_fit_dumb}')

start_pred_dumb = time.perf_counter()
dumb_pred = dummy.predict(val_df[features])
end_pred_dumb = time.perf_counter()
time_pred_dumb = end_pred_dumb - start_pred_dumb
print(f'pred took {time_pred_dumb}')

print(classification_report(val_df[target], dumb_pred,
          target_names=['Delay < 15', 'Delay >= 15']))
print(f'ROC-AUC: {roc_auc_score(val_df[target], dummy.predict_proba(val_df[features])[:, 1]):.4f}')
print(f'accuracy: {accuracy_score(val_df[target], dumb_pred):.4f}')

fit took 0.7585468460001721
pred took 1.6762221999997564


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

  Delay < 15       0.79      1.00      0.88    779753
 Delay >= 15       0.00      0.00      0.00    204004

    accuracy                           0.79    983757
   macro avg       0.40      0.50      0.44    983757
weighted avg       0.63      0.79      0.70    983757

ROC-AUC: 0.5000
accuracy: 0.7926
